In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ex 3.1

The dataset contains quarterly numbers of overnight trips in different regions of Australia, between 1998 and 2018. The trips are categorised according to the purpose of the trip. Find out the region with the greatest mean across these years, where the purpose of the trips was business. Plot the time series representing business trips to this region, showing the mean and standard deviation on the plot.

Complete the solution by writing your code instead of "???".

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/australia_tourism.csv",
                 parse_dates=["date_time"], index_col="date_time")
df.info()

In [ ]:
largest_mean = 0
largest_mean_region = None

for region in df["Region"].unique():
    tdf = df[(df["Region"] == region) & (df["Purpose"] == "Business")]
    ???

print(f"Region with largest mean: {largest_mean_region} ({largest_mean:.2f})")

In [ ]:
# select the business trips for the region with the largest mean
tdf = df[(df["Region"] == ???) & (df["Purpose"] == "Business")].copy()

# add columns for the mean and the upper and lower bounds of the 95% confidence interval
tdf.loc[:,"Mean"] = ???
tdf.loc[:, "2*Stdev"] = tdf["Mean"] + 2 * tdf["Trips"].std()
tdf.loc[:, "-2*Stdev"] = tdf["Mean"] - 2 * tdf["Trips"].std()

# plot the lines with appropriate styles
tdf.plot(y=["Trips", "Mean", "2*Stdev", "-2*Stdev"], 
         figsize=(12, 3), 
         style={"Trips": "blue", "Mean": "r", "2*Stdev": "--r", "-2*Stdev": "--r"}, 
         legend=False,
         use_index=True)

# Ex 3.2

Plot the ACF and PACF plots of the time series from Ex.3.1. Does the series look more like white noise or red noise? What other insights can you get from these plots?

Complete the solution by writing your code instead of "???".

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

???

# Ex 3.3

The US macroeconomic dataset (available in the sktime package) contains values for indicators such as GDP, inflation, and interest rate from 1959 to 2009. Find out which of them are stationary and which are not, using the ADF test and the significance level of 0.05. Find out the series with the strongest evidence of stationarity, plot it as well as its ACF plot. Similarly, find out the series with the strongest evidence of non-stationarity, plot the series and its ACF plot.

Description of the variables can be found [here](https://www.statsmodels.org/dev/datasets/generated/macrodata.html).

You may need to first install the `sktime` package, by running this command in a separate call:

`!pip install sktime`

Complete the solution by writing your code instead of "???".

In [ ]:
from sktime.datasets.forecasting import Macroeconomic

df = Macroeconomic().load()[1]

df.info()

In [ ]:
from statsmodels.tsa.stattools import adfuller

stationary_example = None
nonstationary_example = None

pvals = {}

for col in df.columns:
    adf_stat, pval, lags, obs, crit_vals, icbest = adfuller(???, regression="n")
    pvals[col] = pval

# print the column with the smallest p-value
stationary_example = min(pvals, key=pvals.get)
print(f"Column with smallest p-value: {stationary_example} (p-value: {pvals[stationary_example]:.3f})")

# print the column with the greatest p-value
nonstationary_example = max(pvals, key=pvals.get)
print(f"Column with greatest p-value: {nonstationary_example} (p-value: {pvals[nonstationary_example]:.3f})")

In [ ]:
# Stationary series
???

In [ ]:
# Non-stationary series
???

# Ex 3.4

The dataset contains quarterly numbers of overnight trips in different regions of Australia, between 1998 and 2018. The code below converts the data into a pivot table, where the rows contain dates, the rows contain regions, and cells contain the number of trips to those regions in the given quarters. Your task is to find strong leading indicators of business trips to Adelaide Hills. Perform the following steps:

1. Find out which columns represent stationary time series.
2. Measure the cross-correlation between "Adelaide Hills" each stationary series.
3. Describe significant correlations.

Most of the solution is written, you need to supply your code instead of the question marks ("???").

In [ ]:
# Load the data
df = pd.read_csv("https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/australia_tourism.csv",
                 parse_dates=["date_time"])

In [ ]:
# Select data on business trips
df = df[df["Purpose"] == ???]

# Pivot so each region is a column, each date is a row
pivot = df.pivot_table(index="???", aggfunc="sum",
                       columns='???', values='???',
                       observed=False)
pivot.head(3)

In [ ]:
# find non-stationary columns using the sign level of 0.1 and drop them

from statsmodels.tsa.stattools import adfuller

drop_cols = []

for c in pivot.columns:
    adf_stat, pval, lags, obs, crit_vals, icbest = adfuller(???, regression="n")
    if pval > ???:
        drop_cols.append(c)

pivot = pivot.drop(drop_cols, axis=1)
pivot.info()

In [ ]:
import statsmodels.api as sm

# the target is Adelaide Hills
xcorrs = sm.tsa.stattools.ccf(???, ???, adjusted=False)
xcorrs[:5]

In [ ]:
def build_ccf(y, x, max_lags=10):
    ccf_lags = sm.tsa.stattools.ccf(y, x, adjusted=False)
    plt.figure(figsize=[8, 3])
    lags = np.arange(max_lags) 
    ax = sns.lineplot(y=ccf_lags[:max_lags], x=lags)
    sign_band = 1.96 / np.sqrt(y.shape[0])
    plt.axhspan(-sign_band, sign_band, alpha=0.2)
    ax.set(xlabel='Lags', ylabel='cross-correlation')

# the target is Adelaide Hills
build_ccf(???, ????)

In [ ]:
# the target is Adelaide Hills
xcorrs = sm.tsa.stattools.ccf(???, ???, adjusted=False)
xcorrs[:5]

In [ ]:
# the target is Adelaide Hills
build_ccf(???, ???)